In [4]:
#ควร Run ข้อมูลหลังตลาดหุ้นไทยปิดคือหลัง 17.00 น. เพื่อให้ข้อมูล Last Price ที่ได้เป็นราคาปิด (Close) ของวันล่าสุดที่แท้จริง

#ข้อควรทราบ

#1.เป็นโปรแกรมหาหุ้นใน SET (ไม่รวม MAI) ที่ตรงกับ Trade Setup : 80/20

#2.โปรแกรมนี้ ใช้ข้อมูลฟรีที่มาจาก Yahoo Finance ทั้งหมด
#แม้ค่าต่าง ๆ จะมีความคลาดเคลื่อนเล็กน้อยหากเทียบกับ Platform อื่นๆ เช่น Tradingview แต่ก็ยังมีความใกล้เคียงกัน
#แต่อย่างไรก็ตาม แนะนำให้ตรวจสอบหุ้นแต่ละตัวที่กรองได้ ด้วยกราฟราคาอีกทีว่าตรงกับ Trade Setup 80:20 จริงหรือไม่

#3.ไม่การันตีว่าหุ้นที่ไม่อยู่ในลิส จะไม่เข้า Trade Setup 80:20 เพียงแต่อาจไม่มีข้อมูลที่ครบถ้วนที่จะสามารถใส่เข้ามาในลิส เช่น ไม่พบค่า %Div Yield หรือ PE เป็นต้น

#4.โปรแกรมนี้อาจมีข้อพกพร่อง หากพบโปรดแจ้งแอดมินที่่ m.me/thelightclassroom

#5.รายชื่อหุ้นใน SET update ล่าสุดแล้วเพราะนำมาจาก Excel หน้านี้ของ SET โดยตรง https://www.set.or.th/th/market/information/securities-list/main

In [5]:
#วิธีใช้

#1.สั่งโปรแกรมให้เริ่มทำงาน ด้วยการไปที่ เมนู  "Runtime" >> เลือกกด "Restart Session and Run all"  หากไม่มีให้กด ให้เลือกกด "Run all" แทนได้เลย

#2.รอโปรแกรมทำงานประมาณ 5 นาที เพื่อรับ ลิสรายชื่อหุ้นที่กรองมาเรียบร้อยแล้ว ในรูปแบบ ไฟล์ Excel

#3.เปิดไฟล์ Excel ที่ได้ จะพบ 2 TABs ที่สำคัญ

#3.1. SCREEN_ALL_CANDIDATES จะแสดงรายชื่อหุ้นทั้งที่มี RSI (14) > 80 ภายใน 75 วันทำการย้อนหลัง
#พร้อมกับแสดงรายละเอียดค่าสำคัญไม่ว่าจะเป็น %Div Yield, PE, EMA10>Close และ EMA10>EMA50

#3.2. SCREENED_DY_PE_EMA จะแสดงรายชื่อหุ้นที่มี Pattern ตรงกับ Trade Setup : 80/20
#โดยการนำลิสหุ้นใน TAB ข้อ 5.1 SCREEN_ALL_CANDIDATES ที่มีเงื่อนไขครบทั้ง 4 มิติ
#คือ %Div Yield > 12%, PE < 5, EMA10>Close  = TRUE และ EMA10>EMA50 = TRUE มาบรรจุใส่ใน TAB นี้

#4. นำลิสหุ้นจาก TAB ข้อ 5.2 "SCREENED_DY_PE_EMA" นำไปวิเคราะห์ต่อด้วยกราฟเพื่อหาจังหวะเข้าซื้อได้เลย


In [6]:
!pip -q install yfinance pandas_ta openpyxl lxml html5lib

import pandas as pd
import requests
from io import StringIO
import time

import pandas_ta as ta
import yfinance as yf
from google.colab import files

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# =========================
# 1) GET SET TICKERS (exclude mai)
# =========================
LIST_URL = "https://www.set.or.th/dat/eod/listedcompany/static/listedCompanies_th_TH.xls"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "th-TH,th;q=0.9,en;q=0.8",
}

r = requests.get(LIST_URL, headers=headers, timeout=60)
r.raise_for_status()

df0 = pd.read_html(StringIO(r.text))[0]

# แถว index=1 คือ header จริง
listed = df0.copy()
listed.columns = listed.iloc[1]
listed = listed.iloc[2:].reset_index(drop=True)

# ใช้เฉพาะ "หลักทรัพย์" + "ตลาด"
listed = listed[["หลักทรัพย์", "ตลาด"]].copy()
listed["หลักทรัพย์"] = listed["หลักทรัพย์"].astype(str).str.strip()
listed["ตลาด"] = listed["ตลาด"].astype(str).str.strip()

df_set = listed[listed["ตลาด"].str.upper().eq("SET")].copy()

tickers_bk = (
    df_set["หลักทรัพย์"]
    .dropna()
    .astype(str).str.strip()
    .unique()
    .tolist()
)
tickers_bk = [t if t.endswith(".BK") else f"{t}.BK" for t in tickers_bk]

print("SET tickers =", len(tickers_bk))
print("Sample:", tickers_bk[:20])

# =========================
# 2) RSI SCAN (ANY RSI>80 in last 75 trading days)
# =========================
PERIOD = "220d"      # เผื่อ EMA50 + lookback
RSI_LEN = 14
THRESH = 80
LOOKBACK_BARS = 75

def chunks(lst, n=120):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

frames = []
for ch in chunks(tickers_bk, 120):
    price_df = yf.download(
        tickers=ch,
        period=PERIOD,
        interval="1d",
        group_by="ticker",
        auto_adjust=True,
        threads=True,
        progress=False
    )
    frames.append(price_df)

raw = pd.concat(frames, axis=1)

rows = []
for t in tickers_bk:
    try:
        if t not in raw.columns.get_level_values(0):
            continue

        close = raw[t]["Close"].dropna()
        if len(close) < RSI_LEN + 10:
            continue

        rsi = ta.rsi(close, length=RSI_LEN).dropna()
        rsi_75 = rsi.tail(LOOKBACK_BARS)
        if len(rsi_75) == 0:
            continue

        rows.append({
            "Ticker": t,
            "Passed_ANY_RSI>80_in_75bars": (rsi_75 > THRESH).any(),
            "RSI_last": float(rsi_75.iloc[-1]),
            "RSI_max_75bars": float(rsi_75.max()),
            "Date_of_max_75bars": str(rsi_75.idxmax())[:10],
            "Bars_close": int(len(close))
        })
    except Exception:
        continue

all_debug = pd.DataFrame(rows)
passed_df = (
    all_debug[all_debug["Passed_ANY_RSI>80_in_75bars"]]
    .sort_values("RSI_max_75bars", ascending=False)
    .reset_index(drop=True)
)

print("Hits (RSI>80 any in 75 bars) =", len(passed_df))
display(passed_df.head(20))

# =========================
# 3) SCREENING ต่อ (DY>5%, PE<12, EMA10>Close, EMA10>EMA50)
# =========================
MIN_DIV_YIELD = 0.05   # fraction: 0.05 = 5%
MAX_PE = 12

def calc_ema_flags_from_raw(raw, ticker_bk, ema_fast=10, ema_slow=50, lookback=220):
    if ticker_bk not in raw.columns.get_level_values(0):
        return None

    close = raw[ticker_bk]["Close"].dropna().tail(lookback)
    if len(close) < ema_slow + 5:
        return None

    ema10 = ta.ema(close, length=ema_fast).dropna()
    ema50 = ta.ema(close, length=ema_slow).dropna()
    if len(ema10) == 0 or len(ema50) == 0:
        return None

    close_last = float(close.iloc[-1])
    ema10_last = float(ema10.iloc[-1])
    ema50_last = float(ema50.iloc[-1])

    pullback = (ema10_last > close_last)   # EMA10 > Close
    uptrend  = (ema10_last > ema50_last)   # EMA10 > EMA50
    return close_last, ema10_last, ema50_last, pullback, uptrend

def to_float(x):
    """แปลงเป็น float; แปลงไม่ได้ return None"""
    if x is None:
        return None
    v = pd.to_numeric(x, errors="coerce")
    if pd.isna(v):
        return None
    try:
        return float(v)
    except Exception:
        return None

def normalize_dividend_yield(dy_raw):
    """
    ทำให้ DY เป็น fraction เสมอ
    - ถ้า Yahoo ส่งมาเป็น percent (เช่น 7.25) -> 0.0725
    - ถ้าส่งมาเป็น fraction (เช่น 0.0725) -> 0.0725
    """
    dy_num = to_float(dy_raw)
    if dy_num is None:
        return None, None  # (dy_frac, unit)

    if dy_num > 1:
        return dy_num / 100.0, "percent"
    else:
        return dy_num, "fraction"

def get_fundamentals_yf(ticker_bk):
    """
    คืนค่า:
      dy_raw, pe_raw
      dy_frac (fraction), dy_unit
      pe_num (float)
    """
    try:
        info = yf.Ticker(ticker_bk).info
        dy_raw = info.get("dividendYield", None)
        pe_raw = info.get("trailingPE", None)

        dy_frac, dy_unit = normalize_dividend_yield(dy_raw)
        pe_num = to_float(pe_raw)

        return dy_raw, pe_raw, dy_frac, dy_unit, pe_num
    except Exception:
        return None, None, None, None, None

screen_rows = []
tickers_passed = passed_df["Ticker"].tolist()

for i, t in enumerate(tickers_passed, 1):
    tech = calc_ema_flags_from_raw(raw, t, ema_fast=10, ema_slow=50, lookback=220)
    if tech is None:
        continue
    close_last, ema10_last, ema50_last, pullback, uptrend = tech

    dy_raw, pe_raw, dy_frac, dy_unit, pe = get_fundamentals_yf(t)

    missing_dy = (dy_frac is None)
    missing_pe = (pe is None)

    # ✅ ใช้ dy_frac เท่านั้นในการเทียบ >5%
    pass_dy = (dy_frac is not None) and (dy_frac > MIN_DIV_YIELD)
    pass_pe = (pe is not None) and (pe < MAX_PE)
    passed_all = pass_dy and pass_pe and pullback and uptrend

    # แปลงเป็น % เพื่อดูง่าย
    dy_pct = None if dy_frac is None else dy_frac * 100.0

    screen_rows.append({
        "Ticker": t,

        "DY_raw": dy_raw,
        "PE_raw": pe_raw,

        "DividendYield_frac": dy_frac,
        "DividendYield_%": dy_pct,
        "DY_unit": dy_unit,

        "TrailingPE": pe,

        "Missing_DY": missing_dy,
        "Missing_PE": missing_pe,

        "Close_last": close_last,
        "EMA10_last": ema10_last,
        "EMA50_last": ema50_last,
        "EMA10>Close (Pullback)": pullback,
        "EMA10>EMA50 (Uptrend)": uptrend,

        "Pass_DY>5%": pass_dy,
        "Pass_PE<12": pass_pe,
        "Passed_ALL (DY>5%, PE<12, EMA)": passed_all
    })

    if i % 25 == 0:
        time.sleep(0.5)

screen_df = pd.DataFrame(screen_rows)

# ชีตผู้ผ่านครบทุกเงื่อนไข
screened_df = (
    screen_df[screen_df["Passed_ALL (DY>5%, PE<12, EMA)"]]
    .copy()
    .sort_values(["DividendYield_%", "TrailingPE"], ascending=[False, True])
    .reset_index(drop=True)
)

print("Screened hits (DY/PE/EMA) =", len(screened_df))
display(screened_df.head(30))

# =========================
# 4) README SHEET
# =========================
readme_df = pd.DataFrame([
    ["📌 OVERVIEW",
     "รายชื่อหุ้นในตลาด SET (ไม่รวม mai) ที่เคยมีค่า RSI(14) > 80 อย่างน้อย 1 วัน ภายใน 75 วันทำการล่าสุด"],
    ["", ""],
    ["🧠 CRITERIA (RSI)",
     "• ใช้ RSI(14)\n• ข้อมูลจาก Yahoo Finance\n• ใช้ 75 วันทำการจริง\n• เงื่อนไข: RSI > 80 อย่างน้อย 1 วัน"],
    ["", ""],
    ["📌 SCREENED_DY_PE_EMA",
     "แท็บ SCREENED_DY_PE_EMA คือผลการกรองต่อจาก RSI>80_ANY_75bars ด้วย:\n"
     "• Dividend Yield > 5%\n• P/E < 12\n• EMA(10) > Close (พักฐาน)\n• EMA(10) > EMA(50) (เทรนด์ขาขึ้น)\n\n"
     "หมายเหตุ: หุ้นที่ไม่มีข้อมูล DY หรือ P/E ใน yfinance จะถูกตัดออก (ถือว่าไม่ผ่าน)"],
    ["", ""],
    ["📊 COLUMN NOTES (SCREENING)",
     "• DY_raw / PE_raw = ค่าดิบจาก yfinance\n"
     "• DividendYield_frac = ค่า DY แบบ fraction (0.05 = 5%) ใช้ตัดสินเงื่อนไข\n"
     "• DividendYield_% = ค่า DY แบบเปอร์เซ็นต์เพื่ออ่านง่าย\n"
     "• DY_unit = บอกว่า yfinance ส่งมาเป็น fraction หรือ percent (ไทยบางตัวส่งมาเป็น percent)\n"
     "• Missing_DY / Missing_PE = ข้อมูลหาย/แปลงไม่ได้"],
], columns=["Item", "Description"])

# =========================
# 5) EXPORT + FORMAT EXCEL
# =========================
out_xlsx = "SET_RSI_over_80_any_in_last_75_trading_days.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    passed_df.to_excel(writer, index=False, sheet_name="RSI>80_ANY_75bars")
    all_debug.to_excel(writer, index=False, sheet_name="Debug_All")
    screened_df.to_excel(writer, index=False, sheet_name="SCREENED_DY_PE_EMA")
    screen_df.to_excel(writer, index=False, sheet_name="SCREEN_ALL_CANDIDATES")
    readme_df.to_excel(writer, index=False, sheet_name="README")

wb = load_workbook(out_xlsx)

header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(bold=True, color="FFFFFF")
wrap = Alignment(vertical="top", wrap_text=True)

def format_sheet(ws):
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center")

    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            if cell.value is not None:
                max_len = max(max_len, len(str(cell.value)))
            cell.alignment = wrap
        ws.column_dimensions[col_letter].width = min(max_len + 3, 60)

for name in ["RSI>80_ANY_75bars", "Debug_All", "SCREENED_DY_PE_EMA", "SCREEN_ALL_CANDIDATES", "README"]:
    format_sheet(wb[name])

wb.save(out_xlsx)
files.download(out_xlsx)


SET tickers = 702
Sample: ['2S.BK', '3BBIF.BK', 'A.BK', 'A5.BK', 'AAI.BK', 'AAV.BK', 'ACC.BK', 'ACE.BK', 'ACG.BK', 'ADVANC.BK', 'ADVICE.BK', 'AE.BK', 'AEONTS.BK', 'AFC.BK', 'AGE.BK', 'AH.BK', 'AHC.BK', 'AI.BK', 'AIE.BK', 'AIMCG.BK']
Hits (RSI>80 any in 75 bars) = 166


,Ticker,Passed_ANY_RSI>80_in_75bars,RSI_last,RSI_max_75bars,Date_of_max_75bars,Bars_close
0,CIMBT.BK,True,100.000000,100.000000,2026-05-11,219
1,GLAND.BK,True,100.000000,100.000000,2026-03-09,219
2,LRH.BK,True,100.000000,100.000000,2026-05-01,219
3,F&D.BK,True,51.178096,100.000000,2025-08-07,60
4,NEW.BK,True,21.613153,99.997126,2026-05-15,218
5,TR.BK,True,99.995666,99.995666,2026-06-12,219
6,M-STOR.BK,True,44.619590,97.615273,2026-06-10,219
7,BIZ.BK,True,53.588122,97.330837,2026-06-08,220
8,ML.BK,True,78.113326,95.949703,2026-06-04,220
9,CPTREIT.BK,True,84.867174,95.793843,2026-05-26,216


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcno

Screened hits (DY/PE/EMA) = 19


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcno

,Ticker,DY_raw,PE_raw,DividendYield_frac,DividendYield_%,DY_unit,TrailingPE,Missing_DY,Missing_PE,Close_last,EMA10_last,EMA50_last,EMA10>Close (Pullback),EMA10>EMA50 (Uptrend),Pass_DY>5%,Pass_PE<12,"Passed_ALL (DY>5%, PE<12, EMA)"
0,SCP.BK,11.90,8.136363,0.1190,11.90,percent,8.136363,False,False,8.950000,8.976660,8.633884,True,True,True,True,True
1,UP.BK,10.64,9.031008,0.1064,10.64,percent,9.031008,False,False,23.299999,23.527177,23.047193,True,True,True,True,True
2,METCO.BK,10.20,5.328467,0.1020,10.20,percent,5.328467,False,False,292.000000,293.222506,282.478318,True,True,True,True,True
3,ALLY.BK,9.35,6.555555,0.0935,9.35,percent,6.555555,False,False,4.720000,4.722097,4.590473,True,True,True,True,True
4,JDF.BK,9.23,11.411765,0.0923,9.23,percent,11.411765,False,False,1.940000,1.960099,1.840330,True,True,True,True,True
5,BIZ.BK,8.77,8.560606,0.0877,8.77,percent,8.560606,False,False,5.650000,5.748875,5.297213,True,True,True,True,True
6,AMATAR.BK,8.77,8.841463,0.0877,8.77,percent,8.841463,False,False,7.250000,7.283089,7.055376,True,True,True,True,True
7,SO.BK,8.23,9.387754,0.0823,8.23,percent,9.387754,False,False,4.600000,4.627638,4.536356,True,True,True,True,True
8,GABLE.BK,8.08,6.807018,0.0808,8.08,percent,6.807018,False,False,3.880000,3.969174,3.884587,True,True,True,True,True
9,PCC.BK,8.03,8.787879,0.0803,8.03,percent,8.787879,False,False,2.900000,2.945885,2.835031,True,True,True,True,True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>